# final six model experiment

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Six final training-domain models

Compare FLAMINGO, TNG and equal-domain combined training using dynamics and dynamics plus stellar sum. This notebook shows data, splits, scaling, objective, training, evaluation and paired uncertainty. Read [the design record](../../experiment_design.ipynb) before running.

Default `MODE='prepare'` executes all local audit cells and saves an executed preparation record. `MODE='train'` runs exactly one configuration on pinned Roihu; `MODE='evaluate'` consumes an explicit six-run registry. Missing scientific results are reported as pending, never inferred from an old model.


In [ ]:
MODE='prepare'
CONFIG_NAME='flamingo_dynamics'
MODEL_REGISTRY=''

In [ ]:
import os, sys, json, hashlib, pickle
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
candidates = [Path(os.environ.get('LG_PROJECT', Path.cwd())), Path.cwd(), *Path.cwd().parents]
PROJECT = next(p.resolve() for p in candidates if (p/'src/lg_npe').is_dir())
sys.path.insert(0, str(PROJECT/'src'))
from lg_npe.final_design import *
from lg_npe.registry import environment_record, write_json
from lg_npe.features import feature_names
from lg_npe.evaluation import posterior_metrics
from lg_npe.inference import posterior_draws
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Project:', PROJECT)

## Data and target

Reuse the v3 loaders and their global complete-case population. The target mixes common-host M200c with summed M200c for different-host pairs. Stellar sum is all-bound catalogue stellar mass, with residual HBT-HERONS/SUBFIND definition dependence. No observational noise or domain indicator is added.

The historical FLAMINGO holdout is checked against saved indices. TNG uses the historical split implementation, verified separately against `analogues/splits.py`. New validation groups come only from outer-training groups (seed 43). Every partition is exhaustive and component-disjoint. The holdouts have been analysed previously; this is a controlled follow-up, not untouched prospective confirmation.


In [ ]:
frames = load_final_frames(PROJECT)
splits = final_splits(PROJECT, frames)
population_rows = []
for domain, frame in frames.items():
    for part in ['train','validation','test']:
        ix=splits[domain][part]; selected=frame.iloc[ix]
        population_rows.append(dict(domain=domain,split=part,n=len(ix),
            same_host=int(selected.same_host.sum()),different_host=int((~selected.same_host).sum()),
            components=len(np.unique(splits[domain]['groups'][ix])),
            target_min=selected.log_m200c.min(),target_max=selected.log_m200c.max()))
display(pd.DataFrame(population_rows))
print('Target:', TARGET_DEFINITION)

In [ ]:
BUNDLE = PROJECT/'data/derived/final_experiment_v1'
BUNDLE.mkdir(parents=True,exist_ok=True)
arrays={f'{d}_{k}':v for d,parts in splits.items() for k,v in parts.items()}
identity={d:frame_digest(f) for d,f in frames.items()}
identity['historical_split_sha256']=sha256(PROJECT/HISTORICAL_SUM/'split_indices.npz')
split_path=BUNDLE/'splits.npz'; identity_path=BUNDLE/'identity.json'
if split_path.exists() or identity_path.exists():
    if not (split_path.exists() and identity_path.exists()):
        raise RuntimeError('Incomplete frozen split bundle; inspect before rerunning')
    assert json.loads(identity_path.read_text()) == identity, 'Frozen data identities changed'
    with np.load(split_path) as saved:
        for k,v in arrays.items(): np.testing.assert_array_equal(saved[k],v)
else:
    np.savez_compressed(split_path,**arrays)
    write_json(identity_path,identity)
print('Frozen split bundle:', BUNDLE)

## Six configurations and effective domain contribution

Combined training minimizes the mean of the two domain log losses. Each 64-row batch contains 32 draws from each domain; repeated TNG rows add no independent information. Single-domain training uses 64 draws from that domain. The scaler uses the same domain weighting, only on training rows. The BoxUniform is a support restriction; empirical catalogue target frequencies enter training.

The combined conditional is proportional to `p_F(theta,x)+p_T(theta,x)`, so its domain mixture weights depend on x. The scalar affine MAF cannot represent every multimodal conditional mixture. Calibration and widths must be interpreted with that limitation.


In [ ]:
from lg_npe.controlled_training import validate_final_config, train_controlled_npe
configs={p.stem:json.loads(p.read_text()) for p in sorted((PROJECT/'configs/final').glob('*.json'))}
assert len(configs)==6
design_rows=[]
for name,c in configs.items():
    validate_final_config(c)
    names=feature_names('dynamics',c['mass_feature'])
    domains=c['training_domains']
    X=np.concatenate([frames[d].iloc[splits[d]['train']][names].to_numpy(float) for d in domains])
    labels=np.concatenate([np.repeat(d,len(splits[d]['train'])) for d in domains])
    weights=equal_domain_weights(labels)
    scaler=fit_training_scaler(X,labels)
    effective={d:float(weights[labels==d].sum()) for d in domains}
    batch=balanced_batch_indices(labels,64,np.random.default_rng(42))
    assert all(np.sum(labels[batch]==d)==64//len(domains) for d in domains)
    design_rows.append(dict(model=name,features=names,n_train=len(X),weights=effective,
        scaler_mean=scaler.mean_.tolist(),noise='none'))
display(pd.DataFrame(design_rows))

## Training configuration

Architecture is fixed: MAF, hidden width 64, five transforms, learning rate 1.6488693479340492e-4, batch 64. The historical unused `num_layers` is not activated. New fits repair row-based internal validation for all six models. There is no six-way hyperparameter search.

The reusable optimizer contains only the training loop. It uses equal-domain minibatch NLL, exact full-validation domain means, Adam, clipping at 5, patience 20 and at most 200 epochs. Epoch length is ceil(largest domain training count/64); update and exposure counts are saved. Test rows are not passed to it. The sbi density builder and posterior wrapper are included. A separate synthetic Roihu smoke test must pass before this training mode is launched.


In [ ]:
OUTPUT=PROJECT/'out/final_six'/('preparation' if MODE=='prepare' else MODE)/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
pd.DataFrame(population_rows).to_csv(OUTPUT/'split_counts.csv',index=False)
pd.DataFrame(design_rows).to_csv(OUTPUT/'design.csv',index=False)
audit=dict(status='prepared_no_new_models',mode=MODE,data_identity=identity,
    split_sha256=sha256(BUNDLE/'splits.npz'),target=TARGET_DEFINITION,
    environment=environment_record(),configs=configs,
    source_sha256={str(p.relative_to(PROJECT)):sha256(p) for p in sorted((PROJECT/'src/lg_npe').glob('*.py'))},
    design_sha256=sha256(PROJECT/'out/final_stage_audit/portability_v3/original_design.txt'))
write_json(OUTPUT/'preparation_manifest.json',audit)
assert MODE in ['prepare','train','evaluate']
print('Audit saved:', OUTPUT)
if MODE=='prepare':
    print('PENDING: six new model fits and final metrics; see cluster/roihu/FINAL_STAGE_README.md')

In [ ]:
if MODE=='train':
    require_saved_model_environment()
    c=configs[CONFIG_NAME]; names=feature_names('dynamics',c['mass_feature'])
    def training_arrays(part):
        rows=[frames[d].iloc[splits[d][part]] for d in c['training_domains']]
        return (np.concatenate([f[names].to_numpy(float) for f in rows]),
                np.concatenate([f.log_m200c.to_numpy(float) for f in rows]),
                np.concatenate([np.repeat(d,len(f)) for d,f in zip(c['training_domains'],rows)]))
    train=training_arrays('train'); validation=training_arrays('validation')
    posterior,density,scaler,history=train_controlled_npe(train,validation,c)
    RUN=PROJECT/c['output_root']/c['experiment_id']/STAMP
    RUN.mkdir(parents=True,exist_ok=False)
    with (RUN/'model.pkl').open('xb') as stream:
        pickle.dump(dict(posterior=posterior,density_estimator=density,scaler=scaler,
            feature_names=names,target='log_m200c',config=c),stream)
    write_json(RUN/'run_manifest.json',dict(config=c,data_identity=identity,
        split_sha256=sha256(BUNDLE/'splits.npz'),training_summary=history,
        model_sha256=sha256(RUN/'model.pkl'),environment=environment_record(),
        source_sha256=audit['source_sha256'],design_sha256=audit['design_sha256']))
    pd.DataFrame(history['history']).to_csv(RUN/'training_history.csv',index=False)
    print('Completed model:',RUN)

## Common held-out evaluation

Set `MODEL_REGISTRY` to a JSON object mapping the six configuration names above to their exact run directories. The notebook rejects missing models, wrong configurations and changed splits/data. Metrics use 500 posterior draws and medians. Save row identities, draws and per-pair summaries so every table can be reconstructed. Empirical rank CDFs and coverage are held-out diagnostics, not formal prior-predictive SBC.


In [ ]:
metrics=[]; summaries={}; model_provenance={}
if MODE=='evaluate':
    require_saved_model_environment()
    if not MODEL_REGISTRY: raise RuntimeError('Supply an explicit MODEL_REGISTRY JSON path')
    registry=json.loads(Path(MODEL_REGISTRY).read_text())
    assert set(registry)==set(configs), 'Registry must contain exactly six models'
    import torch
    for label,rel in registry.items():
        run=PROJECT/rel; manifest=json.loads((run/'run_manifest.json').read_text())
        assert manifest['config']==configs[label]
        assert manifest['data_identity']==identity
        assert manifest['split_sha256']==sha256(BUNDLE/'splits.npz')
        assert manifest['model_sha256']==sha256(run/'model.pkl')
        model=load_model(run/'model.pkl'); names=model['feature_names']
        assert names==feature_names('dynamics',configs[label]['mass_feature'])
        model_provenance[label]=dict(run=str(run),model_sha256=manifest['model_sha256'])
        for domain in frames:
            ix=splits[domain]['test']; frame=frames[domain].iloc[ix]
            X=model['scaler'].transform(frame[names].to_numpy(float))
            torch.manual_seed(261001)
            samples=posterior_draws(model['posterior'],X,500,256,'cuda')
            truth=frame.log_m200c.to_numpy()
            summary=posterior_summary(samples,truth,frame.pair_index.to_numpy(),
                splits[domain]['groups'][ix],frame.same_host.to_numpy())
            summaries[label,domain]=summary
            prefix=f'{label}_{domain}'
            summary.to_csv(OUTPUT/f'{prefix}_summary.csv.gz',index=False)
            np.savez_compressed(OUTPUT/f'{prefix}_draws.npz',samples=samples,pair_index=frame.pair_index)
            subsets={'pooled':np.ones(len(frame),bool)}
            if domain=='tng': subsets.update(same_host=frame.same_host.to_numpy(),different_host=~frame.same_host.to_numpy())
            for subset,mask in subsets.items():
                metrics.append(dict(model=label,domain=domain,subset=subset,
                    **posterior_metrics(samples[:,mask],truth[mask])))
    pd.DataFrame(metrics).to_csv(OUTPUT/'six_model_metrics.csv',index=False)
    display(pd.DataFrame(metrics))

## Paired performance uncertainty and figures

Compare combined against each single domain within a feature representation, and stellar sum against dynamics within each training domain. Resample connected components with replacement, identically for both predictions, then stratify hosts. Intervals are conditional on these fitted models; they exclude retraining, volume/cosmic variance and catalogue-definition uncertainty. Different-host TNG has only 48 held-out rows; prioritize RMSE and bias over R2.


In [ ]:
if MODE=='evaluate':
    contrasts=[(f'{d}_{f}',f'combined_{f}') for f in ['dynamics','stellar_sum'] for d in ['flamingo','tng']]
    contrasts += [(f'{d}_dynamics',f'{d}_stellar_sum') for d in ['flamingo','tng','combined']]
    intervals=[]
    for a,b in contrasts:
        for domain in frames:
            ci,raw=paired_component_intervals(summaries[a,domain],summaries[b,domain],repeats=2000)
            ci['model_A']=a; ci['model_B']=b; ci['domain']=domain; intervals.append(ci)
            raw.to_csv(OUTPUT/f'paired_{a}_vs_{b}_{domain}.csv.gz',index=False)
    pd.concat(intervals).to_csv(OUTPUT/'paired_component_intervals.csv',index=False)
    results=pd.DataFrame(metrics)
    fig,axes=plt.subplots(1,3,figsize=(14,4),layout='constrained')
    for ax,metric in zip(axes,['rmse','bias','coverage68']):
        for (domain,subset),f in results.groupby(['domain','subset']):
            ax.plot(f.model,f[metric],marker='o',label=f'{domain}: {subset}')
        ax.set_ylabel(metric); ax.tick_params(axis='x',rotation=70)
        if metric=='coverage68': ax.axhline(.68,color='k',ls=':')
        if metric=='bias': ax.axhline(0,color='k',ls=':')
    axes[0].legend(fontsize=7)
    fig.savefig(OUTPUT/'six_model_comparison.png',dpi=220)
    fig.savefig(OUTPUT/'six_model_comparison.svg')
    plt.show()
    write_json(OUTPUT/'evaluation_manifest.json',dict(status='complete',models=model_provenance,
        data_identity=identity,split_sha256=sha256(BUNDLE/'splits.npz'),posterior_draws=500,
        bootstrap_repeats=2000,seed=261001,environment=environment_record()))

## Interpretation to complete after Roihu

Do not treat preparation as a result. Inspect domain-specific bias, coverage and widths together: lower RMSE alone is insufficient. Combined training includes labelled TNG data and is not zero-shot transfer. Any improvement is conditional on the selected empirical mixture and fixed density family. Neither failure nor improvement identifies feedback or halo finding as a cause. Methods/Results prose must use the completed tables and retain the small different-host sample limitation.
